In [12]:
import pandas as pd
import numpy as np


In [13]:
# Load RUCA dataset
ruca_path = "/Users/priyankaramachandran/Desktop/UCD Courses/CN Project/dataset/Urbanicity 2020 Dataset.csv"

ruca = pd.read_csv(ruca_path)
ruca.head()

,ZIPCode,State,ZIPCodeType,POName,PrimaryRUCA,SecondaryRUCA
0,1,AK,ZIP Code Area,N Dillingham,10,10.0
1,2,AK,ZIP Code Area,Yukon Flats Nat Wildlife,10,10.0
2,3,AK,ZIP Code Area,Alaska Peninsula NWR,10,10.0
3,4,AK,ZIP Code Area,W Kenai Peninsula Borough,10,10.0
4,5,AK,ZIP Code Area,N Lake and Peninsula Borough,10,10.0


In [14]:
# Standardize column names
ruca.columns = ruca.columns.str.lower().str.strip()
ruca.columns

Index(['zipcode', 'state', 'zipcodetype', 'poname', 'primaryruca',
       'secondaryruca'],
      dtype='object')

In [15]:
# Keep useful columns only
keep_cols = [
    "zipcode",
    "state",
    "poname",
    "primaryruca",
    "secondaryruca"
]

ruca = ruca[keep_cols].copy()

ruca.head()

,zipcode,state,poname,primaryruca,secondaryruca
0,1,AK,N Dillingham,10,10.0
1,2,AK,Yukon Flats Nat Wildlife,10,10.0
2,3,AK,Alaska Peninsula NWR,10,10.0
3,4,AK,W Kenai Peninsula Borough,10,10.0
4,5,AK,N Lake and Peninsula Borough,10,10.0


In [16]:
# Clean ZIP codes
ruca["zipcode"] = (
    ruca["zipcode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

ruca["state"] = ruca["state"].astype(str).str.strip()

ruca.head()

,zipcode,state,poname,primaryruca,secondaryruca
0,00001,AK,N Dillingham,10,10.0
1,00002,AK,Yukon Flats Nat Wildlife,10,10.0
2,00003,AK,Alaska Peninsula NWR,10,10.0
3,00004,AK,W Kenai Peninsula Borough,10,10.0
4,00005,AK,N Lake and Peninsula Borough,10,10.0


In [17]:
# Convert RUCA columns to numeric
ruca["primaryruca"] = pd.to_numeric(
    ruca["primaryruca"],
    errors="coerce"
)

ruca["secondaryruca"] = pd.to_numeric(
    ruca["secondaryruca"],
    errors="coerce"
)

ruca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41146 entries, 0 to 41145
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   zipcode        41146 non-null  object 
 1   state          41146 non-null  object 
 2   poname         41146 non-null  object 
 3   primaryruca    41146 non-null  int64  
 4   secondaryruca  41146 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 1.6+ MB


In [18]:
# Filter California only
ruca = ruca[ruca["state"] == "CA"].copy()

ruca.shape

(2610, 5)

In [19]:
# Greater Sacramento metro ZIP codes (60)
sac_zipcodes = [
    "94571",
    "95608", "95610", "95615", "95621", "95624", "95626", "95628",
    "95630", "95632", "95638", "95639", "95640", "95641",
    "95652", "95655", "95660", "95661", "95662", "95670",
    "95671", "95673", "95678", "95680", "95683", "95690", "95693",
    "95742", "95757", "95758", "95762",
    "95811", "95814", "95815", "95816", "95817", "95818", "95819", "95820",
    "95821", "95822", "95823", "95824", "95825", "95826", "95827", "95828",
    "95829", "95830", "95831", "95832", "95833", "95834", "95835", "95836",
    "95837", "95838", "95841", "95842", "95843", "95864",
]

ruca = ruca[
    ruca["zipcode"].isin(sac_zipcodes)
].copy()

ruca.shape

(24, 5)

In [20]:
# Create urbanicity labels (USDA-style bucketing)
# RUCA 1 = metropolitan core, 2-3 = metro commuting, 4+ = non-metro/rural
def classify_ruca(ruca_code):
    if pd.isna(ruca_code):
        return np.nan
    elif ruca_code == 1:
        return "Urban"
    elif ruca_code <= 3:
        return "Suburban"
    else:
        return "Rural"

ruca["urbanicity"] = ruca["primaryruca"].apply(classify_ruca)

print(ruca["urbanicity"].value_counts())
ruca.head()

urbanicity
Urban       22
Rural        1
Suburban     1
Name: count, dtype: int64


,zipcode,state,poname,primaryruca,secondaryruca,urbanicity
39157,95608,CA,Carmichael,1,1.0,Urban
39161,95612,CA,Clarksburg,1,1.0,Urban
39169,95620,CA,Dixon,4,4.0,Rural
39172,95624,CA,Elk Grove,1,1.0,Urban
39174,95626,CA,Elverta,1,1.0,Urban


In [21]:
# Keep final useful columns
final_cols = [
    "zipcode",
    "poname",
    "primaryruca",
    "urbanicity"
]

ruca = ruca[final_cols].copy()

ruca.head()

,zipcode,poname,primaryruca,urbanicity
39157,95608,Carmichael,1,Urban
39161,95612,Clarksburg,1,Urban
39169,95620,Dixon,4,Rural
39172,95624,Elk Grove,1,Urban
39174,95626,Elverta,1,Urban


In [22]:
from pathlib import Path

output_dir = Path("../cleaned_dataset")
output_dir.mkdir(parents=True, exist_ok=True)

ruca.to_csv(output_dir / "urbanicity_sacramento_cleaned.csv", index=False)

print("Saved cleaned Urbanicity data to cleaned_dataset/urbanicity_sacramento_cleaned.csv")

Saved cleaned Urbanicity data to cleaned_dataset/urbanicity_sacramento_cleaned.csv
